# 06 - Expected credit loss (Stage 1)

12-month expected credit loss for the whole portfolio (training and test sets combined, 307,511 loans), using the validated PD model.

**ECL = PD × LGD × EAD**

**Assumptions**
- **PD:** 12-month PD from the logistic regression scorecard. TARGET measures early payment difficulties rather than the regulatory default definition, so PD is treated as a proxy for the 12-month PD.
- **LGD:** fixed at 45%. The data contains no post-default recovery information, so LGD cannot be modelled. Real LGDs for unsecured consumer loans are often higher; sensitivity is tested in the stress test.
- **EAD:** the loan amount (AMT_CREDIT). For revolving loans, a bank would add a credit conversion factor (CCF) on the undrawn limit.
- **Stage:** all loans are treated as Stage 1 (performing), since the data reflects the application date. Staging is added in the next step.
- **Discounting:** not applied, since the 12-month horizon makes its effect small.

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

import joblib
import pandas as pd
from src.woe import load_binners, woe_transform
from src.ecl import compute_ecl, LGD_UNSECURED

train = pd.read_parquet("../data/processed/train.parquet")
test = pd.read_parquet("../data/processed/test.parquet")
portfolio = pd.concat([train, test], ignore_index=True)

binners = load_binners("../models/binners.joblib")
saved = joblib.load("../models/pd_model.joblib")
model, FINAL_VARS = saved["model"], saved["final_vars"]

X = woe_transform(portfolio, binners)[FINAL_VARS]
portfolio["PD"] = model.predict_proba(X)[:, 1]
portfolio["EAD"] = portfolio["AMT_CREDIT"]
portfolio["ECL"] = compute_ecl(portfolio["PD"], portfolio["EAD"])

total_ead = portfolio["EAD"].sum()
total_ecl = portfolio["ECL"].sum()
print(f"Loans: {len(portfolio):,}")
print(f"Total EAD: {total_ead:,.0f}")
print(f"Total 12-month ECL: {total_ecl:,.0f}")
print(f"Coverage ratio (ECL / EAD): {total_ecl / total_ead:.2%}")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loans: 307,511
Total EAD: 184,207,084,196
Total 12-month ECL: 5,974,216,466
Coverage ratio (ECL / EAD): 3.24%


**Portfolio result:** total 12-month ECL is about 6.0 billion on an exposure of about 184 billion, a coverage ratio of 3.24%.

A simple estimate, 8.07% average PD × 45% LGD, would give about 3.6%. Our result coverage ratio of 3.24% is lower because ECL is weighted by loan size: 3.24% ÷ 45% means the exposure-weighted PD is about 7.2%, below the 8.07% average. In other words, larger loans have lower PDs on average.

That matches our model: AMT_GOODS_PRICE has a negative coefficient, so bigger loans get lower predicted risk, probably the same selection effect we found earlier. Large loans were likely approved only for stronger clients.

## Where does the ECL come from

Break it down by PD decile, to see how concentrated the expected losses are

In [3]:
portfolio["PD_DECILE"] = pd.qcut(portfolio["PD"], 10, labels=False) + 1
by_decile = portfolio.groupby("PD_DECILE").agg(
    loans=("PD", "size"),
    mean_pd=("PD", "mean"),
    ead=("EAD", "sum"),
    ecl=("ECL", "sum"),
)
by_decile["ecl_share"] = by_decile["ecl"] / by_decile["ecl"].sum()
by_decile["coverage"] = by_decile["ecl"] / by_decile["ead"]
print(by_decile.style.format({
    "mean_pd": "{:.2%}", "ead": "{:,.0f}", "ecl": "{:,.0f}",
    "ecl_share": "{:.1%}", "coverage": "{:.2%}",
}).to_string())

 loans mean_pd ead ecl ecl_share coverage
PD_DECILE      
1 30754 1.28% 24,479,284,414 137,700,447 2.3% 0.56%
2 30754 2.25% 21,402,429,322 216,025,979 3.6% 1.01%
3 30746 3.10% 19,917,634,761 277,866,441 4.7% 1.40%
4 30751 4.02% 19,141,561,809 345,884,268 5.8% 1.81%
5 30751 5.09% 18,424,432,742 421,648,524 7.1% 2.29%
6 30751 6.40% 17,603,537,787 506,510,938 8.5% 2.88%
7 30751 8.09% 16,808,354,262 611,975,131 10.2% 3.64%
8 30751 10.46% 16,249,671,130 764,517,344 12.8% 4.70%
9 30751 14.39% 15,443,624,340 999,308,682 16.7% 6.47%
10 30751 25.65% 14,736,553,628 1,692,778,712 28.3% 11.49%



**ECL concentration:** expected losses are strongly concentrated in the riskiest loans. The riskiest PD decile holds 10% of the loans and 8% of the exposure, but 28.3% of the ECL; the three riskiest deciles together account for 58%. Coverage rises from 0.56% in the safest decile to 11.49% in the riskiest. Exposure falls as risk rises (24.5 billion in the safest decile vs 14.7 billion in the riskiest), confirming that larger loans carry lower PDs in this portfolio.

## Save the result for staging and the stress test

In [4]:
portfolio.to_parquet("../data/processed/portfolio_ecl.parquet", index=False)
by_decile.to_csv("../models/ecl_by_decile.csv")
print("Saved portfolio with PD, EAD and ECL")

Saved portfolio with PD, EAD and ECL
